In [ ]:
# add libs 

# Monte Carlo Simulation for Agile Estimation
In the previous notebook `01_problem_definition` it got established that point estimates systematicaly fail when applied to work with inherent variance. Using real integration data from my team's Jira history, the analysis showed that the standard 14-day commitment succeeds only 5.1% of the time against historical data. The actual distribution of completion times spans a wide range and places the point estimate entirely outside the standard deviation band. 

The data cleaning exercise added a further clarity: even the underlying dataset required scrutiny! Misclassified tickets were silently pulling the mean down by 7 days, making the original 20% apparent success rate an artefact of poor data hygiene rather than real performance. Once corrected, the picture became very different: more integrations exceed 60 days (10.3%) than meet the 14-day promise (5.1%)!

The conclusion that followe was not that estimation is impossible but rather that a single point estimate is the wrong tool when dealing with high variance. What is needed is a forecasting approach that reflects uncertainty honestly, uses empirical data rather than assumptions, and produces outputs that stakeholders can act on.

This notebook puts that approach to the test. Monte Carlo simulation is selected because it shifts from deterministic promises to probabilistic forecasts. It samples directly from observed history, outputs probability distributions rather than false certainty, and scales naturally to multiple work items addressing the compounding uncertainty that point estimates ignore entirely as the team rarely works on 1 integration at a time.

The question this notebook explores is not whether Monte Carlo simulation can produce more realistic delivery forecasts y modeling historical throughput/cycle time data as probability distribution, rather than point estimates. It will also explore the significance of various parameters which can be used to improve the accuracy  and under what conditions its assumptions break down.

## What is Monte Carlo Simulation?
At its core, Monte Carlo simulation is about using randomness to understand uncertainty. Instead of trying to calculate a single "correct" answer, it asks: *If we let this process play out thousands of times, what does the full range of possible outcomes look like?*

The technique was born in the 1940s when physicist Stanislaw Ulam stuck at home recovering from an illness and playing solitaire started wondering what the probability of winning a game was. Rather than working it out analytically, he thought: Why not just play it out hundreds of times and count? He brought the idea to John von Neumann and they applied it to nuclear physics problems at Los Alamos, and named it after the Monte Carlo Casino in Monaco because at its heart, the method runs on chance.

The insight that made it powerful then is the same one that makes it useful for software delivery forecasting today: *Some problems are too complex or too uncertain to solve with a formula, but perfectly tractable if you just simulate them enough times.*

### The Math Behind Monte Carlo
If you repeat an experiment $N$ times and count how often outcome $A_k$ occurs ($N_k$ times), probability is simply the long-run frequency of that outcome:

$$p_k = \lim_{N \to \infty} \left( \frac{N_k}{N} \right)$$

Where:
- $N$: the total number of times you run the experiment (e.g. number of simulation runs)
- $N_k$: how many times outcome $k$ occurred across all runs
- $p_k$: the probability of outcome $k$, which stabilises as $N$ grows large

This is the foundation Monte Carlo is built on. By running a simulation thousands of times and counting outcomes, we construct probabilities from scratch - no assumptions needed. Once we have probabilities, the next natural question is: *What is the average outcome we should expect?* This is captured by the expectation value. For a random variable $x$ taking values $x_i$ with probabilities $p_i$:

$$\langle x \rangle \equiv E(x) \equiv \sum_i p_i x_i$$

Where:
- $x_i$: a specific outcome (e.g. the total days to finish an integration in one simulation run)
- $p_i$: the probability of that outcome occurring
- $\langle x \rangle$: the true long-run average, also written $E(x)$

More usefully, the function of the outcomes like the total integration duration being above or below a deadline would be:

$$\langle g(x) \rangle \equiv E(g(x)) = \sum_i p_i \, g(x_i)$$

Where:
- $g(x_i)$:  any quantity we want to track per simulation run (e.g. whether the run finished before the target date)
- $p_i \, g(x_i)$: each outcome's contribution, weighted by how likely it is
- $\langle g(x) \rangle$: the weighted average of that quantity across all possible outcomes

In the context of agile forecasting, this is exactly what we compute: the probability that the integration finishes before a given date, averaged across thousands of simulated futures.

#### Variance - Measuring How Spread Out the Outcomes Are
The expected value alone does not tell the full story. Two integrations could have the same average duration but wildly different spreads - one predictable, one chaotic. The **variance** captures this spread:

$$\text{var}(x) = \langle (x - \langle x \rangle)^2 \rangle = \langle x^2 \rangle - \langle x \rangle^2$$

Where:
- $x$: any individual simulation outcome (e.g. total days to complete the integration)
- $\langle x \rangle$: the average outcome across all simulations
- $(x - \langle x \rangle)^2$: the squared distance of each outcome from the average
- $\text{var}(x)$: the average of those squared distances; larger means more unpredictable

The square root of the variance, $\sigma = \sqrt{\text{var}(x)}$, is the **standard deviation** - the most intuitive measure of spread, in the same units as the input data (days in my case). 

This matters enormously for delivery forecasting. High variance means the difference between P50 and P95 forecast is large - which is exactly the honest uncertainty range that point estimates hide entirely.

#### How Monte Carlo Actually Works
In practice, we never know the true probabilities $p_i$ for all possible outcomes - that is exactly the problem we are trying to solve. Monte Carlo gets around this by replacing the theoretical average with a **sample average** computed from $N$ simulation runs.

The **sample mean estimator** is:

$$\bar{A} = \frac{1}{n} \sum_{i=1}^{n} A_i$$

Where:
- $A_i$: the outcome recorded on simulation run $i$ (e.g. total days to finish the integration in my case)
- $n$: the number of simulation runs performed
- $\bar{A}$: the Monte Carlo estimate: the average outcome across all runs

Applied to computing an integral - the general form of most estimation problems this becomes the "crude method" estimator, where $N$ random inputs are drawn and the function $f(x)$ is evaluated at each one:

$$\hat{y} = \frac{1}{N} \sum_{i=1}^{N} f(x_i)$$

Where:
- $x_i$: a random input drawn for simulation run $i$ (e.g. a cycle time sampled from Jira history)
- $f(x_i)$: the outcome computed from that input (e.g. the total integration duration that run produces)
- $N$: the number of simulation runs
- $\hat{y}$: the estimate of the expected outcome, which gets more accurate as $N$ grows

The Central Limit Theorem (CLT) states that, under appropriate conditions, the distribution of a normalized version of the sample mean converges to a standard normal distribution. This holds even if the original variables themselves are not normally distributed. Thus, CLT guarantees that as $N$ grows, $\bar{A}$ is distributed according to a Gaussian centred on the true value, regardless of the shape of the underlying distribution. This is why the method works even when Jira cycle times are skewed or heavy-tailed.

So, this prompts the next question: *How wrong could our estimate be?* The answer comes directly from the standard error of the sample mean:

$$\text{error} = \frac{\sigma}{\sqrt{n}}$$

Where:
- $\sigma$: the standard deviation of the individual simulation run outcomes
- $n$: the number of simulation runs performed
- $\text{error}$: how far estimate $\bar{A}$ is likely to be from the true answer

Two things follow directly from this formula:

1. This error rate does not depend on how complex the problem is - only
on $\sigma$ and $n$. Improvement of Monte Carlo accuracy is possible not just in principle but also in practice simply by running the simulation longer.

2. The error shrinks as $1/\sqrt{n}$, which is slow. To halve the error you need four times as many runs. To reduce error by a factor of 10, you need 100
times as many runs, which visualized looks like:

| Simulation runs $n$ | Error reduces by |
|:-------------------:|:----------------:|
| 100| baseline|
| 400| 2×|
| 10,000| 10×|
| 1,000,000| 100×|

In practice, $n = 10{,}000$ runs is the standard sweet spot - the simulation error becomes negligible, and the real bottleneck shifts entirely to the **quality and size of your historical input data**, not the number of runs.

In [ ]:
# Build the simulation

## Validating the Simulation
...

In [ ]:
# Validation

In [ ]:
# Test simulation fora single integration

### Review results of single integration simulation
...

In [1]:
# Scale the simulation for multiple integrations

### Review results of multiple integrations simulation
...

## Analyse Limitations

## Conclution

## Resources
[1]. David P. Landau, Kurt Binder - A Guide to Monte Carlo Simulations in Statistical Physics (2014, Cambridge University Press) - https://www.eng.uc.edu/~beaucag/Classes/AdvancedMaterialsThermodynamics/Books/David%20P.%20Landau,%20Kurt%20Binder%20-%20A%20Guide%20to%20Monte%20Carlo%20Simulations%20in%20Statistical%20Physics%20(2014,%20Cambridge%20University%20Press)%20-%20libgen.lc.pdf